In [0]:
import os, re, sys
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "src"))
from databricks.sdk import WorkspaceClient
from lh import publish

dbutils.widgets.dropdown("env", "dev", ["dev", "test", "prod"])
env = dbutils.widgets.get("env")
root = os.path.dirname(os.getcwd())
bi = publish.load_bi(root)
api = publish.Api(WorkspaceClient())
folder = api.home() + bi["folder"][1:] if bi["folder"].startswith("~/") else bi["folder"]
want = {(e["group_name"], e["permission_level"]) for e in publish.acl(bi, env)}
results = []


def check(name, ok, detail=""):
    results.append(ok)
    print("PASS" if ok else "FAIL", name, detail)


for name, _ in publish.dashboards(root, env):
    title = f"{name}_{env}"
    did = api.find(f"{folder}/{env}/{title}.lvdash.json")
    check(f"{title}: exists in ~/lh_dashboards/{env}", did is not None)
    if not did:
        continue
    pub = api.published(did)
    check(f"{title}: published", bool(pub.get("revision_create_time")))
    check(f"{title}: viewers query as themselves", pub.get("embed_credentials", False) is False)
    text = api.call("GET", f"/api/2.0/lakeview/dashboards/{did}").get("serialized_dashboard", "")
    other = sorted({m for m in re.findall(r"\b(?:bhg|fqf)_(dev|test|prod)\.", text) if m != env})
    check(f"{title}: reads only {env} catalogs", not other and f"_{env}." in text, other or "")
    acl = api.get_acl(did).get("access_control_list", [])
    direct = {(a["group_name"], p["permission_level"]) for a in acl if a.get("group_name")
              for p in a.get("all_permissions", []) if not p.get("inherited")}
    check(f"{title}: group permissions exactly as config/bi.yml", direct == want,
          "" if direct == want else f"missing {sorted(want - direct)} extra {sorted(direct - want)}")
    check(f"{title}: no access for all workspace users", not any(a.get("group_name") == "users" for a in acl))

print(f"\n{sum(results)}/{len(results)} PASS")
assert all(results), "t27_bi_publish failed"